In [2]:
import polars as pl



#load the data from season 2023
pbp = pl.read_parquet("../data/pbp_2023.parquet")



In [4]:
print(pbp.columns)

['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam', 'side_of_field', 'yardline_100', 'game_date', 'quarter_seconds_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'game_half', 'quarter_end', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'time', 'yrdln', 'ydstogo', 'ydsnet', 'desc', 'play_type', 'yards_gained', 'shotgun', 'no_huddle', 'qb_dropback', 'qb_kneel', 'qb_spike', 'qb_scramble', 'pass_length', 'pass_location', 'air_yards', 'yards_after_catch', 'run_location', 'run_gap', 'field_goal_result', 'kick_distance', 'extra_point_result', 'two_point_conv_result', 'home_timeouts_remaining', 'away_timeouts_remaining', 'timeout', 'timeout_team', 'td_team', 'td_player_name', 'td_player_id', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining', 'total_home_score', 'total_away_score', 'posteam_score', 'defteam_score', 'score_differential', 'posteam_score_post', 'defteam_score_post', 'score_differential

In [8]:
cols = ["game_id", "home_team", "away_team", "posteam", "score_differential", "game_seconds_remaining", "down", "ydstogo","result"]
pbp.select(cols).head(20)

game_id,home_team,away_team,posteam,score_differential,game_seconds_remaining,down,ydstogo,result
str,str,str,str,f64,f64,f64,f64,i32
"""2023_01_ARI_WAS""","""WAS""","""ARI""",null,null,3600.0,null,0.0,4
"""2023_01_ARI_WAS""","""WAS""","""ARI""","""WAS""",0.0,3600.0,null,0.0,4
"""2023_01_ARI_WAS""","""WAS""","""ARI""","""WAS""",0.0,3600.0,1.0,10.0,4
"""2023_01_ARI_WAS""","""WAS""","""ARI""","""WAS""",0.0,3570.0,2.0,7.0,4
"""2023_01_ARI_WAS""","""WAS""","""ARI""","""WAS""",0.0,3535.0,3.0,1.0,4
…,…,…,…,…,…,…,…,…
"""2023_01_ARI_WAS""","""WAS""","""ARI""","""ARI""",0.0,3248.0,2.0,10.0,4
"""2023_01_ARI_WAS""","""WAS""","""ARI""","""ARI""",0.0,3208.0,3.0,13.0,4
"""2023_01_ARI_WAS""","""WAS""","""ARI""","""ARI""",0.0,3165.0,4.0,8.0,4


In [9]:
pbp["game_id"].n_unique()
pbp["game_id"].unique().sort()


game_id
str
"""2023_01_ARI_WAS"""
"""2023_01_BUF_NYJ"""
"""2023_01_CAR_ATL"""
"""2023_01_CIN_CLE"""
"""2023_01_DAL_NYG"""
…
"""2023_20_KC_BUF"""
"""2023_20_TB_DET"""
"""2023_21_DET_SF"""


In [11]:
pbp.filter(pl.col("posteam").is_null()).select(cols + ["desc"]).head(20)

game_id,home_team,away_team,posteam,score_differential,game_seconds_remaining,down,ydstogo,result,desc
str,str,str,str,f64,f64,f64,f64,i32,str
"""2023_01_ARI_WAS""","""WAS""","""ARI""",null,null,3600.0,null,0.0,4,"""GAME"""
"""2023_01_ARI_WAS""","""WAS""","""ARI""",null,null,2803.0,null,0.0,4,"""Timeout #1 by ARI at 01:43."""
"""2023_01_ARI_WAS""","""WAS""","""ARI""",null,null,1915.0,null,0.0,4,"""Timeout #1 by WAS at 01:55."""
"""2023_01_ARI_WAS""","""WAS""","""ARI""",null,null,1901.0,null,0.0,4,"""Timeout #2 by WAS at 01:41."""
"""2023_01_ARI_WAS""","""WAS""","""ARI""",null,null,1819.0,null,0.0,4,"""Timeout #3 by WAS at 00:19."""
…,…,…,…,…,…,…,…,…,…
"""2023_01_BUF_NYJ""","""NYJ""","""BUF""",null,null,111.0,null,0.0,6,"""Timeout #2 by BUF at 01:51."""
"""2023_01_BUF_NYJ""","""NYJ""","""BUF""",null,null,35.0,null,0.0,6,"""Timeout #1 by NYJ at 00:35."""
"""2023_01_BUF_NYJ""","""NYJ""","""BUF""",null,null,29.0,null,0.0,6,"""Timeout #3 by BUF at 00:29."""


In [ ]:
plays = pbp.filter(pl.col("posteam").is_not_null())

print(pbp.height - plays.height) #checkow how many rows I dropped

2638


In [22]:
plays = plays.filter(pl.col("result") != 0)

#see if the team with postion won
home_won = (pl.col("posteam") == pl.col("home_team")) & (pl.col("result") > 0)
away_won = (pl.col("posteam") == pl.col("away_team")) & (pl.col("result") < 0) #result is calc home score - away score so pos means home won, neg means away team won
plays = plays.with_columns(
    pl.when(home_won | away_won).then(1).otherwise(0).alias("posteam_won")
)

plays.select(["game_id", "home_team", "away_team", "posteam", "result", "posteam_won"]).head(20)
plays["posteam_won"].mean()

0.5054330490994535

In [23]:
features = ["score_differential", "game_seconds_remaining", "down", "ydstogo",
            "yardline_100", "posteam_timeouts_remaining", "defteam_timeouts_remaining"]
plays.select(features).null_count()

score_differential,game_seconds_remaining,down,ydstogo,yardline_100,posteam_timeouts_remaining,defteam_timeouts_remaining
u32,u32,u32,u32,u32,u32,u32
0,5,5102,0,847,0,0


In [24]:
plays.filter(pl.col("down").is_null()).select(features + ["play_type", "desc"]).head(20)

score_differential,game_seconds_remaining,down,ydstogo,yardline_100,posteam_timeouts_remaining,defteam_timeouts_remaining,play_type,desc
f64,f64,f64,f64,f64,f64,f64,str,str
0.0,3600.0,null,0.0,35.0,3.0,3.0,"""kickoff""","""5-M.Prater kicks 65 yards from…"
6.0,2955.0,null,0.0,15.0,3.0,3.0,"""extra_point""","""6-J.Slye extra point is GOOD, …"
-7.0,2955.0,null,0.0,35.0,3.0,3.0,"""kickoff""","""6-J.Slye kicks 65 yards from W…"
4.0,2753.0,null,0.0,35.0,3.0,2.0,"""kickoff""","""5-M.Prater kicks 65 yards from…"
4.0,2700.0,null,0.0,null,3.0,2.0,null,"""END QUARTER 1"""
…,…,…,…,…,…,…,…,…
0.0,3600.0,null,0.0,35.0,3.0,3.0,"""kickoff""","""9-G.Zuerlein kicks 65 yards fr…"
-3.0,2885.0,null,0.0,35.0,3.0,3.0,"""kickoff""","""2-T.Bass kicks 61 yards from B…"
-3.0,2700.0,null,0.0,null,3.0,3.0,null,"""END QUARTER 1"""


In [25]:
plays = plays.drop_nulls(subset=features)
plays.select(features).null_count()
print(plays.height)

41925


Basically, I have dropped null values inside the set of features for features like "down" "yardline_100" "game_seconds_renaming". "down" had a lot of nulls due to kickoffs, extra points, anything that isnt a play from scrimmage. In addition I also cleaned data where thier were no rows with posteam, because my model focuses on real time data, so not knowing who has pocession is harmful.